<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP7.2B/MP7.2B_slot_eigenvalues.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP7.2B · The Eigenvalues of a Real Slot

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'slot_problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/")
PROJECT = (['mp72b.py', 'MP7.2B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP7.2B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'slot_problem', 'mp72b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import slot_problem as hs
import mp72b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP7.2B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex07.2's slot_problem.py, by its name in Ex07.2

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex07.2's second notebook showed what the copper's resistance does to a stator slot: it rises 0.39 % per kelvin, so the copper's heat grows with its temperature, and the slot's steady equation becomes a **Helmholtz** equation, $k_\mathrm{eff}\nabla^2\theta + c_1\theta + c_0 = 0$. Its solution grows without bound as $c_1/k_\mathrm{eff}$ approaches the slot's **first eigenvalue** $\lambda_1$ - the smallest $\lambda$ for which $-\nabla^2 u = \lambda u$ has a solution that vanishes on the walls. That sets the **runaway current**, above which the slot has no steady state. The next eigenvalues, $\lambda_2$ and $\lambda_3$, say how fast the slot sheds the parts of a disturbance that vary along it, and their modes are the shapes it takes as the current nears runaway.

The eigenvalues depend only on the **shape** of the slot. Ex07.2's slot is a rectangle, whose eigenvalues are exact. A real slot is not: between teeth with parallel sides the slot widens towards the back of the stator. A designer comparing slot shapes - for another tooth width, another number of slots - wants the eigenvalues of **every taper**. This mini project builds one network that answers a **new taper** in a fraction of the time an eigen-solver needs.

In [ ]:
# The tapered slot of the worked example, beside Ex07.2's rectangle, and its data
fig, ax = plt.subplots(1, 2, figsize=(7.5, 6.4))
mp.draw_slot(ax[0], wb=10e-3); ax[0].set_title("back 10 mm: Ex07.2's rectangle", fontsize=10)
mp.draw_slot(ax[1], wb=12e-3); ax[1].set_title("back 12 mm: opening 8 mm", fontsize=10)
ax[1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout(); plt.show()
rows = [("slot", f"{mp.DEPTH*1e3:.0f} mm deep, area {mp.DEPTH*mp.W_MEAN*1e6:.0f} mm² (mean width {mp.W_MEAN*1e3:.0f} mm), walls at {pb.T_WALL:.0f} °C (Ex07.2)"),
        ("taper", f"width at the back W_b = {mp.WB_RANGE[0]*1e3:.1f} to {mp.WB_RANGE[1]*1e3:.1f} mm, at the opening 20 mm - W_b; straight sides"),
        ("winding", f"{pb.N_WIRES} wires of {pb.WIRE_D*1e3:.0f} mm averaged: k_eff = {pb.K_EFF:.2f} W/(m·K), copper fill {pb.FILL:.3f} (Ex07.2)"),
        ("copper", f"ρ = {pb.RHO_CU:.2e} Ω·m at 20 °C, rising {pb.ALPHA_CU*100:.3f} % per kelvin (Ex07.2)"),
        ("rectangle", "λ = " + ", ".join(f"{l/1e6:.4f}" for l in mp.rectangle_eigenvalues()) + f" per mm², runaway {pb.runaway_current():.2f} A (Ex07.2, exact)")]
for k, v in rows:
    print(f"{k:>10}  {v}")

**What you should see.** Left, Ex07.2's 10 × 20 mm rectangle with its 32 wires; right, the slot of the worked example: 8 mm wide at the opening (the air gap side, at the bottom), 12 mm at the back, 20 mm deep, the rectangle dashed. The area stays 200 mm², so the copper fill and the heat per volume are Ex07.2's; only the shape changes. The rectangle's eigenvalues are 0.1234, 0.1974 and 0.3208 per mm², and its runaway current 158.38 A per wire.

## A2 · What is new against Ex07.2

| | Ex07.2, notebook 2 | this mini project |
|---|---|---|
| the slot | a 10 × 20 mm rectangle | tapered: $W_b$ at the back, $20\ \mathrm{mm} - W_b$ at the opening, 20 mm deep |
| the eigenvalues | exact: $\lambda_{mn} = \pi^2(m^2/W^2 + n^2/H^2)$ | **none exact**: numerical only |
| what is asked | $\lambda_1$ of one slot | $\lambda_1$, $\lambda_2$, $\lambda_3$, their modes and the runaway current for **any taper** |

**Why no exact solution exists.** The rectangle's eigenfunctions are products, $\sin(m\pi x'/W)\,\sin(n\pi y'/H)$, because its walls lie along the coordinate lines and the equation separates. With slanted walls the width depends on the depth, no product of a function of $x$ and a function of $y$ vanishes on them, and the equation does not separate. That is the normal case in engineering.

**The strength of the change.** The taper's strength is $\tau = (W_b - 10\ \mathrm{mm})/10\ \mathrm{mm}$, from 0 to 0.25 over the range. At $\tau = 0$ the slot is Ex07.2's rectangle, and the solver must reproduce its exact eigenvalues. Part C checks it.

## A3 · The equation and its conditions

The eigenpairs $(\lambda_k, u_k)$, $k = 1, 2, 3$, with $\lambda_1 < \lambda_2 < \lambda_3$:

$$-\nabla^2 u_k = \lambda_k\, u_k \quad\text{in the slot}, \qquad u_k = 0 \quad\text{on its four walls}, \qquad \int_\mathrm{slot} u_k^2\,dA = 1$$

| where | condition | why it holds |
|---|---|---|
| the four walls | $u = 0$ | Ex07.2's walls, held at the iron's temperature: a mode of the rise vanishes there |
| the whole slot | $\int u^2\,dA = 1$ | an eigenfunction is fixed only up to a factor; this fixes the factor (and $u = 0$, which satisfies the equation, is excluded) |
| two different modes | $\int u_i u_j\,dA = 0$ | eigenfunctions of different eigenvalues are orthogonal; it keeps the second mode from being the first again |

**The runaway current.** Ex07.2's heat is $q = c_0 + c_1\theta$, with $c_1 = \mathrm{FILL}\,\rho_{20}\,\alpha_\mathrm{Cu}\,J^2$ the growth per kelvin and $J = I/A_\mathrm{wire}$. The steady equation $k_\mathrm{eff}\nabla^2\theta + c_1\theta = -c_0$ has no solution once $c_1 = k_\mathrm{eff}\lambda_1$:

$$I_\mathrm{run} = A_\mathrm{wire}\sqrt{\frac{k_\mathrm{eff}\,\lambda_1}{\mathrm{FILL}\,\rho_{20}\,\alpha_\mathrm{Cu}}}$$

a property of this model, not a rating - real insulation fails far earlier (Ex07.2). It depends on the slot's shape only through $\lambda_1$.

In [ ]:
# The ground truth: the three modes of the 12 mm slot, and how the taper moves the eigenvalues
G = np.load("MP7.2B_truth.npz")
fig, ax = plt.subplots(1, 4, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1, 1, 1.7]})
for k in range(3):
    mp.draw_slot(ax[k], wb=12e-3, modes=G["modes_12mm"][k])
    ax[k].set_title(f"mode {k + 1}: λ = {G['lam_12mm'][k]/1e6:.4f} per mm²", fontsize=9)
W = G["widths"]
for k in range(3):
    ax[3].plot(W * 1e3, [(G[f"lam_{mp.width_key(w)}"][k] / G["rect_lam"][k] - 1) * 100 for w in W], "o-", label=f"λ{k + 1}")
ax[3].axvspan(mp.WB_RANGE[0] * 1e3, mp.WB_RANGE[1] * 1e3, color="grey", alpha=0.12)
ax[3].set_xlabel("width at the back W_b [mm]"); ax[3].set_ylabel("change from the rectangle [%]"); ax[3].legend()
plt.tight_layout(); plt.show()
for w in W:
    k = mp.width_key(w)
    print(f"back {w*1e3:5.2f} mm: λ = " + ", ".join(f"{l/1e6:.4f}" for l in G[f"lam_{k}"]) + f" per mm², runaway {float(G[f'runaway_{k}']):.2f} A")

**What you should see.** The three modes of the 12 mm slot: one hump, two and three along the depth, each one half-wave across, as in the rectangle but pushed towards the wider back. Their eigenvalues are 0.1226, 0.2018 and 0.3253 per mm². Right, how far each eigenvalue is from the rectangle's as the back widens: $\lambda_1$ falls by only 0.6 % at 12 mm, $\lambda_2$ and $\lambda_3$ rise by 2.2 and 1.4 %. The first mode spreads over the whole slot, and widening the back by as much as the opening narrows leaves its eigenvalue almost unchanged: it changes with the square of the taper, not with the taper. The higher modes have a hump of their own in the narrow opening, where a mode must bend more across the slot, and that raises them. The runaway current falls from 158.38 A to 157.90 A at 12 mm. The 9.5 mm slot gives exactly the 10.5 mm slot's eigenvalues: it is the same slot turned upside down (question 7).

## A4 · What is asked

| quantity | at | required |
|---|---|---|
| $\lambda_1$ | the test widths 10.5, 11.25 and 12 mm, which you do not train on | within **0.3 %** of the ground truth |
| $\lambda_2$ and $\lambda_3$ | the same widths | within **1 %** |
| the runaway current | the same widths | within **0.5 A** |
| the time to answer a new width online: the three eigenvalues, the runaway current and the three modes at the reference grid's 861 nodes | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C) |

One network, trained once, offline, takes the width at the back as an input: a new taper is answered by evaluating it, not by solving again. The training time is reported apart, as the offline cost. The modes are reported and compared with the ground truth, but no tolerance is set on them.

**The agreed accuracy.** 0.3 % on $\lambda_1$ is 0.5 A on the runaway current, about 0.3 %: the taper itself moves the runaway current by 0.5 A at 12 mm, so a model must be at least that good to see it. 1 % on $\lambda_2$ and $\lambda_3$ is half of what the taper does to them. The reference model's grid was chosen to meet all of it (part C).

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

$$(x,\ y,\ W_b)\ \longmapsto\ \big(u_1,\ u_2,\ u_3\big)(x, y; W_b), \qquad W_b\ \longmapsto\ \big(\lambda_1,\ \lambda_2,\ \lambda_3\big)(W_b)$$

the three modes at a point of the slot, and the three eigenvalues, for a width at the back. The width is an **input**: a new taper is answered by evaluating the network.

**A map that makes every slot the same square.** The slot's half-width at depth $y$ is $a(y) = 5\ \mathrm{mm}\,(1 + \tau\,y/B)$, with $y$ from $-B$ at the opening to $+B$ at the back, $B$ = 10 mm. The coordinates

$$\xi = \frac{x}{a(y)}, \qquad \eta = \frac{y}{B}$$

map every tapered slot onto the square $-1 \le \xi, \eta \le 1$: the slanted walls become $\xi = \pm 1$. A network that sees $(\xi, \eta, \tau)$ works on one fixed domain for every taper, and `mp.to_xy` maps back.

The scales:

In [ ]:
# The scales: the rectangle's modes, and what the taper does
lam_r = mp.rectangle_eigenvalues()
for k, (m, n) in enumerate(((1, 1), (1, 2), (1, 3))):
    print(f"λ{k + 1} of the rectangle = π²({m}²/W² + {n}²/H²) = {lam_r[k]/1e6:.4f} per mm²: {m} half-wave across, {n} along")
print(f"the 4th is (1, 4): {np.pi**2*(1/mp.W_MEAN**2 + 16/mp.DEPTH**2)/1e6:.4f} per mm²; across twice, (2, 1): {np.pi**2*(4/mp.W_MEAN**2 + 1/mp.DEPTH**2)/1e6:.4f}")
for wb in (10e-3, 12e-3, 12.5e-3):
    print(f"back {wb*1e3:4.1f} mm: τ = {mp.taper(wb):.3f}, opening {(2 * mp.W_MEAN - wb)*1e3:.1f} mm")

**What you should see.** The rectangle's first three eigenvalues belong to modes with one half-wave across and one, two and three along the 20 mm depth; the fourth is (2, 1), two half-waves across, at 0.4195 per mm², then (1, 4) at 0.4935: well above the third, so the three asked are well separated from the rest. The range of tapers runs from $\tau = 0$ (the rectangle) to 0.25 (7.5 mm at the opening, 12.5 mm at the back).

## B2 · The physics the network is trained with

There are two ways to put the eigenproblem into a loss.

**(a) The residual, with trainable eigenvalues** (the PINN of Ex07.2's style). Each mode's residual $-\nabla^2 u_k - \lambda_k u_k$ at points of the slot, with $\lambda_k$ a trainable output; the normalisation $\int u_k^2 = 1$ and the orthogonality $\int u_i u_j = 0$ as loss terms, since without them $u = 0$ makes every residual zero. Nothing in it says *which* eigenpair to find: the training may settle on any, and the order must be imposed.

**(b) The energy: Rayleigh's quotient.** The first eigenvalue is the smallest value of

$$R[u] = \frac{\int |\nabla u|^2\,dA}{\int u^2\,dA}$$

over all functions that vanish on the walls, and the first mode is the function that reaches it; the second is the smallest over functions orthogonal to the first, and so on (the min-max principle). A loss built on $R$ needs only first derivatives, cannot be fooled by $u = 0$ (the quotient does not change with the size of $u$), and finds the eigenpairs **in order**. The integrals are taken by Gauss quadrature on the mapped square, where $dA = a(y)\,B\,d\xi\,d\eta$. With the three modes $u_k$ of the network, the 3 × 3 matrices

$$S_{ij} = \int \nabla u_i\cdot\nabla u_j\,dA, \qquad M_{ij} = \int u_i\,u_j\,dA$$

hold everything: the eigenvalues of $S$ against $M$ (the Ritz values) are upper bounds of $\lambda_1$, $\lambda_2$, $\lambda_3$, equal to them when the three outputs span the three modes.

| loss term | evaluated | enforces |
|---|---|---|
| route (a): $\mathcal{L}_\mathrm{eq}$, $\mathcal{L}_\mathrm{norm}$, $\mathcal{L}_\mathrm{orth}$ | points in the slot, at random widths | the equation with trainable $\lambda_k$, the normalisation, the orthogonality |
| route (b): $\sum_k R_k / \lambda_k^\mathrm{rect}$ after Gram–Schmidt | Gauss points of the mapped square, at random widths | the modes, in order, by the min-max principle |
| $\mathcal{L}_\lambda$ | the same widths | the network's eigenvalues equal to the Ritz values of its modes |

**What can be built in.** The walls hold for any weights if each output is multiplied by $(1-\xi^2)(1-\eta^2)$, which vanishes on all four walls of every taper. The slot is symmetric about $x = 0$, and the three modes asked are even in $\xi$ (one half-wave across). Whether you use the map, the symmetry and which route, is your choice in part D.

## B3 · The training points

| function | returns | used for |
|---|---|---|
| `mp.quadrature(n_xi, n_eta)` | Gauss points $(\xi, \eta)$ on the square and their weights | the integrals of route (b), and the normalisation |
| `mp.to_xy(xi, eta, wb)` | the points $(x, y)$ [m] of the slot of back width `wb` | evaluating the network on the slot |
| `mp.half_width(y, wb)` | $a(y)$ [m] | the area element $a\,B\,d\xi\,d\eta$ |
| `mp.sample_widths(n, seed)` | back widths [m], uniform over 10 to 12.5 mm | the tapers of a step |
| `mp.ref_nodes(wb)` | the reference grid's 21 × 41 nodes [m] | where the modes are compared with the ground truth |

The **test widths, 10.5, 11.25 and 12 mm**, lie inside the range but no sampler chooses them: the network is scored where it was not fitted.

In [ ]:
# The Gauss points mapped onto two slots, and the widths of training
xi, eta, w = mp.quadrature(24, 48)
fig, ax = plt.subplots(1, 3, figsize=(12, 6), gridspec_kw={"width_ratios": [1, 1, 1.5]})
for a, wb in zip(ax[:2], (10e-3, 12.5e-3)):
    x, y = mp.to_xy(xi, eta, wb)
    mp.draw_slot(a, wb=wb, wires=False); a.scatter(x * 1e3, y * 1e3, s=1, c="k", zorder=5)
    area = np.sum(w * mp.half_width(y, wb) * mp.B)
    a.set_title(f"back {wb*1e3:.1f} mm: {xi.size} points\nweights sum to {area*1e6:.2f} mm²", fontsize=9)
ax[2].hist(mp.sample_widths(5000) * 1e3, bins=40); [ax[2].axvline(t * 1e3, color="r") for t in mp.WB_TEST]
ax[2].set_xlabel("width at the back W_b [mm]"); ax[2].set_ylabel("tapers drawn"); ax[2].set_title("training widths; the test widths in red", fontsize=9)
plt.tight_layout(); plt.show()

**What you should see.** The same 1 152 Gauss points on the rectangle and on the most tapered slot: their columns follow the walls, denser towards the walls, as Gauss points are. Their weights times the area element sum to 200.00 mm² on both: the slot's area, to rounding. Right, the widths drawn uniformly from 10 to 12.5 mm, with the three test widths in red.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the eigen-solver an engineer would run, on an engineering grid | the same scheme on grids fine enough to be ten thousand times more exact than anything scored, extrapolated |
| grid | 20 × 40 cells (0.5 mm across at the mean width) | 160 × 320 and 320 × 640 cells (0.0625 and 0.03125 mm), combined by Richardson extrapolation |
| its job | to be **timed** against the network: the online cost of a new taper | to **score** the network and the reference |

## C2 · The scheme, step by step

1. **A grid that follows the walls.** The map of part B takes the slot onto the square; a uniform grid of $n_x \times 2n_x$ cells on the square is a grid of quadrilaterals in the slot whose columns follow the slanted walls, so every wall node lies on a wall.
2. **Bilinear elements.** In each cell the mode is the bilinear interpolation of its four nodes. The cell's stiffness, $\int\nabla N_i\cdot\nabla N_j\,dA$, and mass, $\int N_i N_j\,dA$, are integrated by 2 × 2 Gauss points and added into two sparse matrices $K$ and $M$.
3. **The walls.** The wall nodes are removed: $u = 0$ there.
4. **The eigenproblem** $K\mathbf{u} = \lambda M\mathbf{u}$: the three smallest eigenvalues by shift-and-invert about zero (`scipy.sparse.linalg.eigsh`), one sparse factorisation.
5. **The modes** are normalised to $\int u^2 dA = 1$ and made positive where they are largest.

**Why not Ex07.2's finite differences?** The five-point stencil on a square grid cannot follow a slanted wall: it cuts it into stair steps, and the eigenvalues then converge at first order and irregularly, as the cell below shows. Richardson extrapolation needs a known order; on the wall-following grid it is two.

In [ ]:
# The ground truth: convergence at 12 mm, the zero-strength check, and the stair steps
G = np.load("MP7.2B_truth.npz")
print("cells across x along   λ1 [1/mm²]   λ2 [1/mm²]   λ3 [1/mm²]   solve [s]")
for n, l, s in zip(G["conv_nx"], G["conv_lam"], G["conv_seconds"]):
    print(f"   {int(n):4d} x {2*int(n):<4d}       {l[0]/1e6:.6f}     {l[1]/1e6:.6f}     {l[2]/1e6:.6f}     {s:.2f}")
d = np.abs(np.diff(G["conv_lam"], axis=0))
print("change between grids, relative:", np.array2string(d / G["conv_lam"][1:], precision=1))
print("observed order:", np.round(G["conv_order"], 2), "| extrapolated:", np.round(G["conv_extrapolated"] / 1e6, 6), "per mm²")
print(f"back 10 mm (τ = 0), against Ex07.2's exact rectangle: largest relative difference {float(G['zero_strength_rel']):.1e}")
print("\nfive-point finite differences on a square grid, the walls as stair steps (12 mm):")
for h, l in zip(G["stair_h"], G["stair_lam"]):
    print(f"   spacing {h*1e3:.5f} mm: λ = " + ", ".join(f"{v/1e6:.6f}" for v in l) + f"   (λ1 {(l[0]/G['lam_12mm'][0]-1)*100:+.2f} %)")

**What you should see.** Four grids, each twice as fine. Each eigenvalue changes by about a quarter of its previous change: the **observed order is 2.00** for all three, and the Richardson extrapolation from the last two grids, $\lambda_\mathrm{fine} + (\lambda_\mathrm{fine} - \lambda_\mathrm{coarse})/3$, is within about $10^{-7}$ of the converged value - ten thousand times finer than the 0.3 % asked. At $\tau = 0$ the same code gives Ex07.2's exact rectangle to $4\times10^{-10}$: the code is right where the answer is known. The stair-step finite differences, by contrast, are still 0.36 % low on $\lambda_1$ at 0.03125 mm and gain only about half of their error per halving: first order, from the walls.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest of a sequence (10, 20, 40 cells across) that meets the agreed accuracy: on 10 × 20 cells $\lambda_1$ is 0.70 % off and the runaway current 0.55 A; on **20 × 40**, 0.18 % and 0.14 A. The cell below scores it at the test widths and times a new taper: the grid built, the matrices assembled, the eigenproblem solved and the runaway current computed. It runs on the CPU of the runtime: a sparse solve of this size gains nothing from a GPU.

In [ ]:
# The reference model against the ground truth, at the three test widths
worst = {"lam1": 0.0, "lam23": 0.0, "runaway": 0.0}
for wb in mp.WB_TEST:
    k = mp.width_key(wb)
    r = mp.reference(wb)                                    # 20 x 40 cells
    e = r["lam"] / G[f"lam_{k}"] - 1
    e_run = r["runaway"] - float(G[f"runaway_{k}"])
    worst = {"lam1": max(worst["lam1"], abs(e[0])), "lam23": max(worst["lam23"], np.abs(e[1:]).max()), "runaway": max(worst["runaway"], abs(e_run))}
    print(f"back {wb*1e3:5.2f} mm: λ off by " + ", ".join(f"{v*100:+.3f} %" for v in e) + f"; runaway {r['runaway']:.2f} A, off by {e_run:+.2f} A")
t_ref = middle(lambda: mp.reference(11.7e-3))              # the online cost of a new taper
print(f"\nreference model, 20 x 40 cells: worst λ1 {worst['lam1']*100:.2f} %, worst λ2 or λ3 {worst['lam23']*100:.2f} %, "
      f"worst runaway {worst['runaway']:.2f} A; {t_ref*1e3:.1f} ms per new taper")
print(f"the network must answer a new taper in under {t_ref*1e3/10:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_lam1": worst["lam1"], "worst_lam23": worst["lam23"], "worst_runaway_A": worst["runaway"]}   # kept for part E

**What you should see.** At the three test widths the reference model's eigenvalues are all slightly high - bilinear elements give upper bounds - by 0.18 % ($\lambda_1$), 0.21 % ($\lambda_2$) and 0.39 % ($\lambda_3$), and its runaway current is 0.14 A high: inside the agreed 0.3 %, 1 % and 0.5 A. A new taper takes 7 to 22 ms on one CPU thread in our runs, depending on what else the machine ran: the grid, the assembly of 800 cells and one factorisation of 741 unknowns. The time is kept in `REF`; part E compares the network against it on this same runtime.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, with two calls:

| call | arguments | returns |
|---|---|---|
| `net(x, y, wb)` | a point of the slot $x$, $y$ [m] and the width at the back `wb` [m], each `(n, 1)` | the three modes at the point, `(n, 3)` |
| `net.eigenvalues(wb)` | widths at the back [m], `(n, 1)` | $\lambda_1$, $\lambda_2$, $\lambda_3$ [1/m²], `(n, 3)` |

The modes need not be normalised or orthogonal: part D's `answer` makes them so (by Gram–Schmidt with the matrix $M$) before comparing them with the ground truth. Whatever you compute inside - the map, built-in walls, features, a head for the eigenvalues - stays inside `net`.

## D2 · The loss, given

Both routes of part B. `rayleigh(net, wbs)` returns the matrices $S$ and $M$ of the network's three modes, by Gauss quadrature on the mapped square, for each width of `wbs`: `(len(wbs), 3, 3)` each, in 1/m² and 1 (the modes in 1/m). `residual_terms(net, seed)` returns route (a)'s three terms at random points and widths: the equation with `net.eigenvalues` divided by the rectangle's eigenvalue, the normalisation and the orthogonality.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

LAM_RECT = torch.tensor(mp.rectangle_eigenvalues(), device=DEVICE)   # 1/m², the rectangle's: the scale of each λ_k
_xi, _eta, _w = mp.quadrature(24, 48)
QXI, QETA, QW = T(_xi), T(_eta), T(_w)                    # Gauss points of the mapped square and their weights

def rayleigh(net, wbs, create_graph=True, stiffness=True):
    """S_ij = ∫ ∇u_i·∇u_j dA and M_ij = ∫ u_i u_j dA [1/m², 1] for each width of wbs (m): (len(wbs), 3, 3) each.
    stiffness=False skips S (returned as None) and the derivatives it needs."""
    nq, nw = len(QXI), len(wbs)
    W = T(np.repeat(np.asarray(wbs, float), nq))
    xi, eta = QXI.repeat(nw, 1), QETA.repeat(nw, 1)
    x, y = mp.to_xy(xi, eta, W)                            # the Gauss points on each slot [m]
    x, y = x.detach().requires_grad_(True), y.detach().requires_grad_(True)
    u = net(x, y, W)                                       # (nw nq, 3)
    dA = (QW.repeat(nw, 1) * 0.5 * mp.W_MEAN * (1 + (W - mp.W_MEAN) / mp.W_MEAN * eta) * mp.B).reshape(nw, nq, 1)   # w a(y) B
    if not stiffness:
        u = u.reshape(nw, nq, 3)
        return None, torch.einsum("bqi,bqj->bij", u * dA, u)
    gx = torch.stack([torch.autograd.grad(u[:, k].sum(), x, create_graph=create_graph, retain_graph=True)[0][:, 0] for k in range(3)], 1)
    gy = torch.stack([torch.autograd.grad(u[:, k].sum(), y, create_graph=create_graph, retain_graph=True)[0][:, 0] for k in range(3)], 1)
    u, gx, gy = u.reshape(nw, nq, 3), gx.reshape(nw, nq, 3), gy.reshape(nw, nq, 3)
    M = torch.einsum("bqi,bqj->bij", u * dA, u)
    S = torch.einsum("bqi,bqj->bij", gx * dA, gx) + torch.einsum("bqi,bqj->bij", gy * dA, gy)
    return S, M

def residual_terms(net, seed, n=2000):
    """Route (a): the equation -∇²u_k - λ_k u_k at random points and widths, divided by λ_k of the rectangle
    times the size of u_k; the normalisation ∫u_k² = 1 and the orthogonality, from the Gauss points."""
    rng = np.random.default_rng(seed)
    wb = mp.sample_widths(n, seed)
    x, y = mp.to_xy(rng.uniform(-1, 1, n), rng.uniform(-1, 1, n), wb)
    X, Y, W = T(x).requires_grad_(True), T(y).requires_grad_(True), T(wb)
    u = net(X, Y, W)
    lam = net.eigenvalues(W)                               # (n, 3) [1/m²]
    out = {"equation": 0.0}
    for k in range(3):
        uk = u[:, k:k + 1]
        lap = d(d(uk, X), X) + d(d(uk, Y), Y)
        out["equation"] = out["equation"] + (((-lap - lam[:, k:k + 1] * uk) / (LAM_RECT[k] * 100.0)) ** 2).mean()   # u ~ 1/(10 mm)
    _, M = rayleigh(net, mp.sample_widths(4, seed + 1), create_graph=True)
    eye = torch.eye(3, device=DEVICE)
    out["norm"] = ((torch.diagonal(M, dim1=1, dim2=2) - 1) ** 2).mean()
    out["orthogonality"] = (((M - eye) * (1 - eye)) ** 2).sum((1, 2)).mean()
    return out

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`, honouring the contract above. Decide:

* **what the network sees.** $x$, $y$ and the width scaled; or the mapped $(\xi, \eta)$ of part B and the taper $\tau$;
* **what is built in.** The walls, the symmetry about $x = 0$;
* **how it gives the eigenvalues.** Trainable numbers per width - a small network of the width - or computed from the modes;
* **its size.** Layers, neurons, activation (tanh, or another smooth one: the derivatives must exist).

In [ ]:
# TODO 1 - your network. net(x, y, wb) -> (n, 3) modes; net.eigenvalues(wb) -> (n, 3) [1/m²]
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, x, y, wb):
        ...                                              # the three modes at (x, y), shape (n, 3)

    def eigenvalues(self, wb):
        ...                                              # λ1, λ2, λ3 [1/m²], shape (n, 3)

torch.manual_seed(72)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with route (a), route (b) or both. Decide the optimiser or optimisers, the steps, the learning rate and its schedule, how many widths a step sees, the weight of each term. Keep the history of the loss for TODO 3, and **measure the offline time**: set `TRAIN_SECONDS`.

`quick_score()` below gives, over the three test widths, the worst relative errors of $\lambda_1$, $\lambda_2$ and $\lambda_3$ from `net.eigenvalues` [%] and the worst error of the runaway current [A], from the ground truth; `answer(wb)` is how part E asks your network: the eigenvalues, the runaway current, and the three modes at the reference grid's nodes, normalised and made orthogonal with $M$.

In [ ]:
# A quick score, given: the worst errors at the test widths
G = np.load("MP7.2B_truth.npz")
_xi_n, _eta_n = np.meshgrid(np.linspace(-1, 1, mp.REF_NX + 1), np.linspace(-1, 1, 2 * mp.REF_NX + 1))
_xi_q, _eta_q, _w_q = mp.quadrature(8, 16)               # Gauss points enough to normalise the modes
XI_A, ETA_A = T(np.r_[_xi_n.ravel(), _xi_q]), T(np.r_[_eta_n.ravel(), _eta_q])   # the 861 nodes, then 128 Gauss points
W_Q, N_NODES = T(_w_q), _xi_n.size

def answer(wb):
    # The network's eigenvalues [1/m²], runaway current [A] and modes at the reference grid's nodes (3, 41, 21)
    # [1/m] for one back width wb [m]: the modes made orthonormal with M (Gram-Schmidt) and positive where largest.
    with torch.no_grad():
        W1 = torch.full((1, 1), wb, device=DEVICE)
        lam = net.eigenvalues(W1)[0]
        x, y = mp.to_xy(XI_A, ETA_A, W1)                     # the nodes and the Gauss points on this slot [m]
        u = net(x, y, W1.expand(len(x), 1))                  # (861 + 128, 3), one call
        uq = u[N_NODES:]
        dA = W_Q * mp.half_width(y[N_NODES:], W1) * mp.B     # Gauss weight times the area element [m²]
        L = torch.linalg.cholesky((uq * dA).T @ uq)          # M = L Lᵀ, from the Gauss points
        v = torch.linalg.solve_triangular(L, u[:N_NODES].T, upper=False).T   # Gram-Schmidt: orthonormal, in order
        v = v * torch.sign(v.gather(0, v.abs().argmax(0, keepdim=True)))
    lam = lam.cpu().numpy()
    return lam, float(mp.runaway(lam[0])), v.T.reshape(3, *_xi_n.shape).cpu().numpy()

def quick_score():
    worst = np.zeros(4)
    for wb in mp.WB_TEST:
        k = mp.width_key(wb)
        lam, run, _ = answer(wb)
        worst = np.maximum(worst, np.r_[np.abs(lam / G[f"lam_{k}"] - 1) * 100, abs(run - float(G[f"runaway_{k}"]))])
    return tuple(worst)

In [ ]:
# TODO 2 - your training
history = {}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst errors: λ1 %.3f %%, λ2 %.3f %%, λ3 %.3f %%, runaway %.3f A" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the three modes of the 12 mm slot next to the ground truth's, with their errors. Save each figure you want in the report as `MP7.2B_report_<name>.png` (in `cc.output_path`): part E's PDF embeds them.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the eigenvalues, the runaway current and the modes at the test widths, and outside the range (9.5 and 13 mm);
* a sweep of the eigenvalues over 9 to 13.5 mm;
* the **online time** of a new taper on this runtime - `answer(wb)` for a width it has not been asked before - for 1, 10 and 100 new widths, against the reference model run 1, 10 and 100 times on the same runtime.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"widths": np.array(G["widths"])}
for wb in G["widths"]:
    lam, run, modes = answer(float(wb))
    k = mp.width_key(wb)
    out[f"lam_{k}"], out[f"runaway_{k}"], out[f"modes_{k}"] = lam, np.array(run), modes
sweep = np.linspace(9.0e-3, 13.5e-3, 19)                  # every 0.25 mm
with torch.no_grad():
    out["sweep_wb"], out["sweep_lam"] = sweep, net.eigenvalues(T(sweep)).cpu().numpy()

# online time: a new taper, the network against the reference model, on this runtime
timing = {}
for n in (1, 10, 100):
    def new_widths(n=n):
        ws = np.random.default_rng().uniform(10.05e-3, 12.45e-3, n)   # widths never asked before
        return [answer(float(w)) for w in ws]
    timing[f"net_{n}"] = middle(new_widths)
    ws = np.linspace(10.1e-3, 12.4e-3, n)
    timing[f"ref_{n}"] = middle(lambda: [mp.reference(float(w)) for w in ws], reps=3 if n == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP7.2B_net.pt"))   # the trained weights, beside the report
print(f"online, one new taper: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's accuracy and time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eight questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP7.2B_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst λ1 {REF['worst_lam1']*100:.2f} %, worst λ2 or λ3 {REF['worst_lam23']*100:.2f} %, "
      f"worst runaway {REF['worst_runaway_A']:.2f} A, {REF['t_ref_s']*1e3:.1f} ms per new taper")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* at each test width: the relative errors of $\lambda_1$, $\lambda_2$ and $\lambda_3$ [%], the error of the runaway current [A], and the largest error of each mode as a share of its peak [%];
* the network's online time for one new taper, the reference model's on the same runtime (both in `R`), and their ratio;
* the offline training time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.3f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP7.2B_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the three eigenvalues against the width at the back from 9 to 13.5 mm, the network's sweep and the ground truth's points, relative to the rectangle's; the runaway current against the width; and the online times of the network and the reference model for 1, 10 and 100 tapers.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eight questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 4 are about your network and its result; 5 to 7 ask you to use it; 8 concludes. The next cell writes `MP7.2B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: what it sees, what is built into its form (the walls, the symmetry, the map), how it gives the eigenvalues, its layers and neurons, and its number of weights. Why this design?": """
""",
    "2. Your training: route (a), route (b) or both, the points or quadrature, how many widths a step sees, the optimiser(s), steps and learning rates, and the offline time in seconds. What was hardest to bring down?": """
""",
    "3. The three eigenvalues and the runaway current at 10.5, 11.25 and 12 mm: give each error against the ground truth. Are they within 0.3 %, 1 % and 0.5 A, and how do they compare with the reference model's?": """
""",
    "4. The modes at the test widths: your largest error of each, as a share of its peak, and where in the slot it sits. Which mode is hardest, and why? Show it in a figure.": """
""",
    "5. The online time for a new taper: your network against the reference model on the same runtime, in milliseconds, and the ratio; then for 10 and 100 tapers. What does each spend its time on?": """
""",
    "6. How much does the taper move each eigenvalue and the runaway current from Ex07.2's rectangle, from 10 to 12.5 mm, by your network and by the ground truth? Explain from the modes why λ1 hardly moves while λ2 and λ3 rise.": """
""",
    "7. Outside the range: your errors at 13 mm and at 9.5 mm. The 9.5 mm slot is the 10.5 mm slot turned upside down and has the same eigenvalues; does your network know it, and how could it be made to?": """
""",
    "8. To conclude: for which design task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? What would it take to answer slots of another depth or with a rounded bottom?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.3f} |" for k, v in METRICS.items()]
    out = ["# MP7.2B — The Eigenvalues of a Real Slot", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP7.2B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP7.2B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP7.2B_report.md into MP7.2B_report.pdf, with any figure
# saved as MP7.2B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP7.2B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP7.2B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP7.2B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP7.2B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass